## Task 3 - Cleaning and analysing 

The cleaning function changes outlying 999 transactions values in the dataframe, to the meadian of valid transactions. 
This is easily done with mathematical operations on numpy arrays. The output is then an array of all cleaned prices.

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
from task_3_cleaning import compute_refund_ratio

# make cleaning function:
def clean_outliers(df: pd.DataFrame) -> np.array:  
    """Changes outlying transactions values in the dataframe, to the meadian of valid transactions.
    
        Arguments:
            df: DataFrame with invalid outlying price values.
    
        Returns:
            Array containing outlier-cleaned transactions.
        """
    # converts price values to array
    cleaned = np.array(df["price"])

    mu = np.mean(cleaned)
    sigma = np.std(cleaned)

    # gather outlying values
    outliers = cleaned > mu + 3*sigma

    valid_prices = cleaned[~outliers]
    median_valid = np.median(valid_prices)

    # replaces all outlier values in cleaned, with the median of the valid prices.
    cleaned[outliers] = median_valid

    return cleaned

The next function calculates the ratio between element-wise refund and sales values.
With sales and refund values as input, this will be used later on, to make an array of ratios in task 4.

In [ ]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates the ratio between element-wise sales and refund values.
    
        Arguments:
            sales_count: Array of the sales count for each element.

            refund_count: Array of the refund count for each element.

        Returns:
            Array containing the refund/sales ratio element-wise.
        """
    ratios = []
    for ref, sale in zip(refund_count, sales_count):
        rat = ref / (sale + 1e-6)
        ratios.append(rat)
    ratios_arr = np.array(ratios)

    return ratios_arr

## Task 4 -  Data analysis using Pandas

The `chrono` function sets the column [timestamp] to pd.datetime so Python handles it as time, 
and sorts it chronologically.

In [ ]:
def chrono(df: pd.DataFrame) -> pd.DataFrame:
    """Creates DataFrame of a list of dictionaries. Sorts the transactions by timestamps.

    Arguments:
        df: DataFrame containing information of transactions.

    Returns:
        DataFrame with transactions in chronological order.
    """
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_values("timestamp")




The next functions returns the total prize for each product within 1-hour windows, 
pivot tables sorting item revenue across status "SUCCESS", "REFUND" and "CANCELLED", 
and calculates the ratio between element-wise refund and sales values in given DataFrame. This last function 
makes use of the `compute_refund_ratio` for this specific datafram.

In [ ]:
def one_hour(a_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Calculates total price for each product within 1-hour windows.

    Arguments:
        a_dataframe: dataframe sorted by timestamp, containing transactions.

    Returns:
        DataFrame sorted by 1-hour windows. One row for each item, within each 1-hour window.
    """   
    return a_dataframe.groupby("item").resample("1h")["price"].sum() 

In [ ]:
def by_status(b_dataframe: pd.DataFrame) -> pd.pivot_table:
    """Creates pivot table, sorting item revenue across status "SUCCESS", "REFUND" and "CANCELLED".

    Arguments:
        b_dataframe: dataframe sorted by timestamp, containing transactions.

    Returns:
        Pivot table.
    """    
    return pd.pivot_table(
        b_dataframe,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

In [ ]:
def use_ratio(c_dataframe: pd.DataFrame) -> np.array:
    """Calculates the ratio between element-wise sales and refund values in given DataFrame.
        
        Arguments:
            c_dataframe: A dataframe containing transactions, either a sale, refund or cancelled transaction.

        Returns:
            Array containing the sales/refund ratio element-wise.
        """
    counts = c_dataframe.groupby(["item", "status"]).size().unstack(fill_value=0)
    success_arr = counts["SUCCESS"].to_numpy()
    refund_arr = counts["REFUND"].to_numpy()

    return compute_refund_ratio(refund_arr, success_arr)